# Phase 2: NLP & Static Clustering

Loads `data/master_dataset.csv`, embeds posts with `sentence-transformers`,
and clusters them with HDBSCAN and DBSCAN as a static (offline) baseline for
trend detection. This baseline is a precursor to the online, time-decayed
micro-clustering built in Phase 3.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.ml_engine.static_clustering import (
    cluster_dbscan,
    cluster_hdbscan,
    cluster_summary,
    reduce_umap,
)
from src.ml_engine.vectorizer import TextVectorizer, load_corpus

DATA_PATH = PROJECT_ROOT / "data" / "master_dataset.csv"

## 1. Load the corpus

Requires `data/master_dataset.csv` to already exist (run the Reddit streamer
or historical loader first).

In [ ]:
df = load_corpus(DATA_PATH)
print(f"Loaded {len(df)} rows")
df.head()

## 2. Embed posts with sentence-transformers

In [ ]:
vectorizer = TextVectorizer()
embeddings = vectorizer.fit_transform(df["text"].tolist())
embeddings.shape

## 3. Reduce dimensionality with UMAP

HDBSCAN and DBSCAN both struggle with the "curse of dimensionality" in raw
384-dim sentence embeddings, so we project down to a smaller space first.

In [ ]:
reduced_embeddings = reduce_umap(embeddings, n_components=5)
reduced_embeddings.shape

## 4. Cluster with HDBSCAN and DBSCAN

Compare the two methods on the same reduced embeddings. `min_cluster_size`
and `eps` are the main knobs to tune as the dataset grows.

In [ ]:
hdbscan_labels = cluster_hdbscan(reduced_embeddings, min_cluster_size=5, min_samples=5)
dbscan_labels = cluster_dbscan(reduced_embeddings, eps=0.5, min_samples=5)


def describe(labels: np.ndarray, name: str) -> None:
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    noise_ratio = float(np.mean(labels == -1))
    print(f"{name}: {n_clusters} clusters, {noise_ratio:.1%} noise")


describe(hdbscan_labels, "HDBSCAN")
describe(dbscan_labels, "DBSCAN")

## 5. Visualize clusters in 2D

Project the embeddings down to 2D purely for visualization (separate from
the 5D projection used for clustering).

In [ ]:
embeddings_2d = reduce_umap(embeddings, n_components=2)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(
    embeddings_2d[:, 0],
    embeddings_2d[:, 1],
    c=hdbscan_labels,
    cmap="tab20",
    s=15,
)
ax.set_title("HDBSCAN clusters (UMAP 2D projection)")
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
plt.colorbar(scatter, ax=ax, label="cluster")
plt.show()

## 6. Inspect sample titles per cluster

Sanity-check that clusters correspond to coherent topics.

In [ ]:
summary = cluster_summary(df, hdbscan_labels, text_column="title", samples_per_cluster=5)
pd.set_option("display.max_colwidth", 100)
summary